# 05 — Hyperparameter Tuning

Use Optuna to find the best XGBoost hyperparameters, then compare against the current baseline (75.4%).

**Approach**
- Same chronological 80/20 train/test split as notebook 04
- TimeSeriesSplit(3) cross-validation **within the training set** to avoid leakage
- Optuna minimises the mean CV log-loss over 100 trials
- Best params evaluated on the held-out test set

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import warnings
import json
import numpy as np
import pandas as pd
import optuna
from sklearn.impute import SimpleImputer
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

PROCESSED = repo_root / 'data' / 'processed'

df = pd.read_parquet(PROCESSED / 'features.parquet')
df['category_men'] = (df['category'] == 'men').astype(int)
print(f'Loaded {len(df)} matches')

## 1. Train / Test Split

In [ ]:
FEATURES = [
    'elo_t1', 'elo_t2', 'elo_diff',
    'win_rate_t1', 'win_rate_t2', 'win_rate_diff',
    'form_streak_t1', 'form_streak_t2', 'form_streak_diff',
    'days_rest_t1', 'days_rest_t2', 'days_rest_diff',
    'level_win_rate_t1', 'level_win_rate_t2', 'level_win_rate_diff',
    'pair_win_rate_t1', 'pair_win_rate_t2', 'pair_win_rate_diff',
    'pair_matches_t1', 'pair_matches_t2',
    'matches_played_t1', 'matches_played_t2', 'matches_played_diff',
    'h2h_wins_t1', 'h2h_wins_t2', 'h2h_total', 'h2h_win_rate_t1',
    'ranking_t1', 'ranking_t2', 'ranking_diff',
    'tournament_level_weight', 'round', 'category_men',
]

df_sorted = df.sort_values('played_at').reset_index(drop=True)
X = df_sorted[FEATURES]
y = df_sorted['target']

split_idx = int(len(df_sorted) * 0.80)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f'Train: {len(X_train)}  |  Test: {len(X_test)}')

## 2. Optuna Study

Each trial proposes a set of hyperparameters, trains on 3-fold time-series CV within the training set, and returns mean log-loss. Optuna uses Bayesian optimisation (TPE sampler) to focus trials on promising regions of the search space.

In [ ]:
imputer = SimpleImputer(strategy='median')
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed  = imputer.transform(X_test)

tscv = TimeSeriesSplit(n_splits=3)

def objective(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 100, 600),
        'max_depth':         trial.suggest_int('max_depth', 2, 8),
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample':         trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight':  trial.suggest_int('min_child_weight', 1, 10),
        'reg_alpha':         trial.suggest_float('reg_alpha', 0.0, 5.0),
        'reg_lambda':        trial.suggest_float('reg_lambda', 0.0, 5.0),
        'eval_metric': 'logloss',
        'random_state': 42,
        'verbosity': 0,
    }
    clf = XGBClassifier(**params)
    scores = cross_val_score(clf, X_train_imputed, y_train,
                             cv=tscv, scoring='neg_log_loss', n_jobs=-1)
    return -scores.mean()

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=100, show_progress_bar=True)

print(f'\nBest log-loss: {study.best_value:.4f}')
print(f'Best params:')
for k, v in study.best_params.items():
    print(f'  {k}: {v}')

## 3. Evaluate Best Model on Test Set

In [ ]:
BASELINE_ACC = 0.754  # XGBoost from notebook 04
BASELINE_AUC = 0.838

best_clf = XGBClassifier(
    **study.best_params,
    eval_metric='logloss',
    random_state=42,
    verbosity=0,
)
best_clf.fit(X_train_imputed, y_train)

y_pred  = best_clf.predict(X_test_imputed)
y_proba = best_clf.predict_proba(X_test_imputed)[:, 1]

acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)

print(f'Tuned XGBoost   acc={acc:.1%}  AUC={auc:.3f}')
print(f'Baseline        acc={BASELINE_ACC:.1%}  AUC={BASELINE_AUC:.3f}')
print()
acc_delta = (acc - BASELINE_ACC) * 100
auc_delta = auc - BASELINE_AUC
print(f'Accuracy delta: {acc_delta:+.1f}pp')
print(f'AUC delta:      {auc_delta:+.3f}')

## 4. Optuna Visualisations

In [ ]:
from optuna.visualization import plot_optimization_history, plot_param_importances

plot_optimization_history(study).show()
plot_param_importances(study).show()

## 5. Save Best Params

In [ ]:
out_path = repo_root / 'src' / 'models' / 'best_params.json'
with open(out_path, 'w') as f:
    json.dump(study.best_params, f, indent=2)
print(f'Saved best params → {out_path}')